In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("/Users/chanokchonkarinrak/Documents/GitHub/AMR_Thesis/data/MDR_summary")
YEAR_MIN, YEAR_MAX = 2015, 2024

TOP_DEFAULT = 5
TOP_ECOLI = 8
ECOLI_NAME = "Escherichia coli"
KLEBSIELLA_NAME = "Klebsiella pneumoniae"

OUT_ROOT = DATA_DIR / "plots_all_orgs"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

need = {"x_year", "organism_full", "Count", "Total_MDR", "Percentage", "MDR_id"}

# =========================
# functions
def safe_name(s: str) -> str:
    return s.replace("/", "-").replace("\\", "-").replace(":", "-").strip()

def plot_one_organism(file_path: Path):
    # -------- Load --------
    df = pd.read_excel(file_path)
    missing = need - set(df.columns)
    if missing:
        print(f"[SKIP] {file_path.name} missing columns: {missing}")
        return

    # -------- Clean --------
    df["x_year"] = pd.to_numeric(df["x_year"], errors="coerce")
    df["Count"] = pd.to_numeric(df["Count"], errors="coerce")
    df["Total_MDR"] = pd.to_numeric(df["Total_MDR"], errors="coerce")
    df["Percentage"] = pd.to_numeric(df["Percentage"], errors="coerce")

    df = df[df["x_year"].between(YEAR_MIN, YEAR_MAX)].copy()
    df = df.dropna(subset=["x_year", "MDR_id"])

    df["x_year"] = df["x_year"].astype(int)
    df["MDR_id"] = df["MDR_id"].astype(str)

    # organism name
    org = str(df["organism_full"].dropna().iloc[0]) if not df["organism_full"].dropna().empty else file_path.stem
    org_safe = safe_name(org)

    # Top N
    top_n = TOP_ECOLI if org == ECOLI_NAME else TOP_DEFAULT

    out_dir = OUT_ROOT / org_safe
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------- Top MDR_id --------
    top_ids = (
        df.groupby("MDR_id")["Count"]
          .sum()
          .sort_values(ascending=False)
          .head(top_n)
          .index
    )

    df_top = df[df["MDR_id"].isin(top_ids)].copy()

    # Pivot Top N
    pv_pct = (
        df_top.pivot_table(index="x_year", columns="MDR_id", values="Percentage", aggfunc="sum")
             .reindex(range(YEAR_MIN, YEAR_MAX + 1))
             .fillna(0.0)
    )
    pv_pct.columns = pv_pct.columns.astype(str)
    pv_pct = pv_pct.loc[:, ~pv_pct.columns.duplicated()]

    # PLOT A: Line trend (Percentage) Top N
    x = pv_pct.index.to_numpy(dtype=int)

    plt.figure(figsize=(12, 6))
    for mdr_id in pv_pct.columns:
        y = pv_pct[mdr_id].to_numpy(dtype=float)
        plt.plot(x, y, marker="o", linewidth=2, label=mdr_id)

    plt.title(f"{org} MDR trend by MDR_id (Top {top_n}) - Percentage")
    plt.xlabel("Year")
    plt.ylabel("Percentage (%)")
    plt.xticks(list(range(YEAR_MIN, YEAR_MAX + 1)))
    plt.grid(True, alpha=0.3)
    plt.legend(title="MDR_id", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()

    out_a = out_dir / f"{org_safe}_top{top_n}_trend_percentage.png"
    plt.savefig(out_a, dpi=200)
    plt.close()

    # PLOT B: total MDR count by year
    year_sum = (
        df.groupby("x_year", as_index=False)
          .agg(total_mdr_count=("Count", "sum"))
    )
    year_sum = (
        year_sum.set_index("x_year")
                .reindex(range(YEAR_MIN, YEAR_MAX + 1))
                .fillna(0.0)
                .reset_index()
    )

    x2 = year_sum["x_year"].to_numpy(dtype=int)
    y2 = year_sum["total_mdr_count"].to_numpy(dtype=float)

    plt.figure(figsize=(10, 5))
    plt.plot(x2, y2, marker="o", linewidth=2)
    plt.title(f"{org} total MDR Count by year (sum across MDR_id)")
    plt.xlabel("Year")
    plt.ylabel("Total MDR Count")
    plt.xticks(list(range(YEAR_MIN, YEAR_MAX + 1)))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    out_c = out_dir / f"{org_safe}_total_mdr_count_by_year.png"
    plt.savefig(out_c, dpi=200)
    plt.close()

    # PLOT C: Heatmap เฉพาะ E. coli & K. pneumoniae
    if org == ECOLI_NAME or KLEBSIELLA_NAME:
        heat = pv_pct.T  # rows=MDR_id, cols=year

        plt.figure(figsize=(12, 0.6 * len(heat) + 3))
        im = plt.imshow(heat.to_numpy(dtype=float), aspect="auto")
        plt.title(f"{org} MDR_id heatmap (Top {top_n}) - Percentage")
        plt.xlabel("Year")
        plt.ylabel("MDR_id")
        plt.xticks(range(len(heat.columns)), [int(c) for c in heat.columns], rotation=45)
        plt.yticks(range(len(heat.index)), heat.index.astype(str))
        plt.colorbar(im, label="Percentage (%)")
        plt.tight_layout()

        out_hm = out_dir / f"{org_safe}_top{top_n}_heatmap_percentage.png"
        plt.savefig(out_hm, dpi=200)
        plt.close()

    print(f"[OK] {file_path.name} -> Top {top_n} | saved in {out_dir.name}")

xlsx_files = sorted(DATA_DIR.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError(f"No .xlsx files found in {DATA_DIR}")

for fp in xlsx_files:
    plot_one_organism(fp)

print("\nDone. All plots saved under:", OUT_ROOT)


[OK] Acinetobacter baumannii.xlsx -> Top 5 | saved in Acinetobacter baumannii
[OK] Enterococcus faecalis.xlsx -> Top 5 | saved in Enterococcus faecalis
[OK] Enterococcus faecium.xlsx -> Top 5 | saved in Enterococcus faecium
[OK] Escherichia coli.xlsx -> Top 8 | saved in Escherichia coli
[OK] Klebsiella pneumoniae.xlsx -> Top 5 | saved in Klebsiella pneumoniae
[OK] Pseudomonas aeruginosa.xlsx -> Top 5 | saved in Pseudomonas aeruginosa

Done. All plots saved under: /Users/chanokchonkarinrak/Documents/GitHub/AMR_Thesis/data/MDR_summary/plots_all_orgs


In [1]:
import pandas as pd

/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv("/Users/chanokchonkarinrak/Documents/GitHub/AMR_Thesis/data/MDR/AllYears_DrugClass_tested.csv")

/var/folders/f0/09md1vw56wn4jjhchxblsy0r0000gn/T/ipykernel_29236/1234822521.py:1: DtypeWarning: Columns (2,7,13,19,25,26,30,31,32,33,34,40,41,42,44,45,46,47,49,51,52,55,56,60,61,62,63,64,65,68,69,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,97,98) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/chanokchonkarinrak/Documents/GitHub/AMR_Thesis/data/MDR/AllYears_DrugClass_tested.csv")


In [4]:
df.head()

,ROW_IDX,COUNTRY_A,X_HOS_CODE,X_HOS_NAME,X_YEAR,LABORATORY,REGION,ORIGIN,PATIENT_ID,SEX,...,X_REFER,NOSOCOMIAL,X_REF_NO,AXS_ND,AMB_ND10,VOR_ND1,VAM_ND,Tested_Classes,Missing_Classes,Missing_Count
0,1,THA,1067200.0,ลำปาง,2015,LPA,1.0,NaN,NaN,m,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"AMINOGLYCOSIDES, CARBAPENEMS, CEPHEMS, FLUOROQ...",NaN,0
1,2,THA,1067200.0,ลำปาง,2015,LPA,1.0,NaN,NaN,m,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"AMINOGLYCOSIDES, CARBAPENEMS, CEPHEMS, FLUOROQ...",NaN,0
2,3,THA,1067200.0,ลำปาง,2015,LPA,1.0,NaN,NaN,f,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"AMINOGLYCOSIDES, CARBAPENEMS, CEPHEMS, FLUOROQ...",NaN,0
3,4,THA,1067200.0,ลำปาง,2015,LPA,1.0,NaN,NaN,m,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"AMINOGLYCOSIDES, CARBAPENEMS, CEPHEMS, FLUOROQ...",NaN,0
4,5,THA,1067200.0,ลำปาง,2015,LPA,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"AMINOGLYCOSIDES, CARBAPENEMS, CEPHEMS, FLUOROQ...",NaN,0


In [10]:
df_aba = df[df["ORGANISM_FULL"] == "Acinetobacter baumannii"].copy()

In [13]:
df_aba.shape

(418049, 102)

In [24]:
(df_aba['meropenem'].isna()).sum()

27737